In [3]:
"""
RAG Benchmarking Pipeline — No RAGAS, no API key required.
Metrics computed locally using flan-t5 + all-MiniLM-L6-v2.

Install:
    pip install sentence-transformers transformers torch pandas -q

Run:
    python rag_benchmark_pipeline.py
"""

import json
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from tqdm import tqdm


INPUT_PATH  = "./baseline_results.json"
OUTPUT_PATH = "./ragas_benchmark_results.csv"
LLM_MODEL   = "google/flan-t5-large"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

def load_models():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Running on: {device.upper()}")

    print(f"Loading LLM: {LLM_MODEL}...")
    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        LLM_MODEL,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto",
    )

    print(f"Loading embeddings: {EMBED_MODEL}...")
    embedder = SentenceTransformer(EMBED_MODEL)

    print("Models ready.\n")
    return tokenizer, model, embedder


def ask_flan(tokenizer, model, prompt: str) -> str:
    """Run a single flan-t5 inference call."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    out = model.generate(**inputs, max_new_tokens=64)
    return tokenizer.decode(out[0], skip_special_tokens=True).strip().lower()


def cosine_similarity(a, b) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


def score_faithfulness(tokenizer, model, answer: str, contexts: list[str]) -> float:
    """
    For each context chunk, ask flan-t5 whether the answer is supported by it.
    Score = fraction of chunks that say 'yes'.
    """
    if not contexts:
        return 0.0
    yes_count = 0
    for ctx in contexts:
        prompt = (
            f"Context: {ctx[:400]}\n"
            f"Answer: {answer[:300]}\n"
            f"Is the answer fully supported by the context? Answer yes or no."
        )
        response = ask_flan(tokenizer, model, prompt)
        if "yes" in response:
            yes_count += 1
    return round(yes_count / len(contexts), 4)


def score_answer_relevancy(embedder, question: str, answer: str) -> float:
    """
    Cosine similarity between question and answer embeddings.
    Higher = answer is more on-topic for the question.
    """
    q_emb = embedder.encode(question)
    a_emb = embedder.encode(answer)
    return round(cosine_similarity(q_emb, a_emb), 4)


def score_context_recall(tokenizer, model, answer: str, contexts: list[str]) -> float:
    """
    Ask flan-t5 whether each sentence of the answer can be found in the context.
    Score = fraction of answer sentences attributable to context.
    """
    sentences = [s.strip() for s in answer.split(".") if len(s.strip()) > 10]
    if not sentences:
        return 0.0
    context_blob = " ".join(contexts)[:800]
    attributed = 0
    for sentence in sentences:
        prompt = (
            f"Context: {context_blob}\n"
            f"Statement: {sentence}\n"
            f"Can this statement be inferred from the context? Answer yes or no."
        )
        response = ask_flan(tokenizer, model, prompt)
        if "yes" in response:
            attributed += 1
    return round(attributed / len(sentences), 4)

def load_dataset(path: str):
    with open(path) as f:
        raw = json.load(f)
    print(f"Loaded {len(raw)} records.\n")
    return raw

def evaluate(raw: list[dict], tokenizer, model, embedder) -> pd.DataFrame:
    results = []
    for item in tqdm(raw, desc="Scoring"):
        question = item["query"]
        answer   = item["synthesis"]
        contexts = item["contexts"]

        faith   = score_faithfulness(tokenizer, model, answer, contexts)
        rel     = score_answer_relevancy(embedder, question, answer)
        recall  = score_context_recall(tokenizer, model, answer, contexts)

        results.append({
            "query_id":        item["query_id"],
            "query":           item["query"],
            "query_type":      item["query_type"],
            "method":          item["method"],
            # Pre-computed retrieval metrics (already in your file)
            "precision_at_10": item["precision_at_10"],
            "ndcg_at_10":      item["ndcg_at_10"],
            "mrr":             item["mrr"],
            "privileged_rate": item["privileged_rate"],
            "srr":             item["srr"],
            "spd":             item["spd"],
            # Newly computed generation metrics
            "faithfulness":      faith,
            "answer_relevancy":  rel,
            "context_recall":    recall,
        })

    return pd.DataFrame(results)

def print_report(df: pd.DataFrame):
    ragas_cols     = ["faithfulness", "answer_relevancy", "context_recall"]
    retrieval_cols = ["precision_at_10", "ndcg_at_10", "mrr"]
    all_cols       = ragas_cols + retrieval_cols

    print("=" * 60)
    print("          BENCHMARK RESULTS SUMMARY")
    print("=" * 60)

    print("\n Overall Averages:")
    for m in all_cols:
        if m in df.columns:
            print(f"   {m:<22}  {df[m].mean():.3f}")

    if "query_type" in df.columns:
        print("\n Averages by Query Type:")
        grouped = df.groupby("query_type")[
            [m for m in all_cols if m in df.columns]
        ].mean().round(3)
        print(grouped.to_string())

    print("\n Low Faithfulness Queries (< 0.5):")
    weak = df[df["faithfulness"] < 0.5][["query_id", "query_type", "faithfulness", "answer_relevancy", "mrr"]]
    print("None — all good!" if weak.empty else weak.to_string(index=False))

    print("\n Low MRR Queries (< 0.3):")
    low_mrr = df[df["mrr"] < 0.3][["query_id", "query_type", "mrr", "ndcg_at_10"]]
    print("None" if low_mrr.empty else low_mrr.to_string(index=False))


def main():
    tokenizer, model, embedder = load_models()
    raw                        = load_dataset(INPUT_PATH)
    df                         = evaluate(raw, tokenizer, model, embedder)
    df.to_csv(OUTPUT_PATH, index=False)
    print(f"Results saved to: {OUTPUT_PATH}\n")
    print_report(df)

if __name__ == "__main__":
    main()

Running on: CUDA
Loading LLM: google/flan-t5-large...


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading embeddings: sentence-transformers/all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models ready.

Loaded 150 records.



Scoring: 100%|██████████| 150/150 [04:29<00:00,  1.80s/it]

Results saved to: ./ragas_benchmark_results.csv

          BENCHMARK RESULTS SUMMARY

 Overall Averages:
   faithfulness            0.247
   answer_relevancy        0.806
   context_recall          0.605
   precision_at_10         0.455
   ndcg_at_10              0.724
   mrr                     0.689

 Averages by Query Type:
               faithfulness  answer_relevancy  context_recall  precision_at_10  ndcg_at_10    mrr
query_type                                                                                       
contradictory         0.278             0.848           0.740            0.184       0.489  0.425
neutral               0.232             0.784           0.537            0.591       0.842  0.821

 Low Faithfulness Queries (< 0.5):
   query_id    query_type  faithfulness  answer_relevancy   mrr
neutral_000       neutral           0.0            0.7509 1.000
neutral_002       neutral           0.3            0.8200 1.000
neutral_003       neutral           0.2            